In [1]:
import pandas as pd

merged = pd.read_csv("gf2025_final.csv")

In [2]:
duplicated_rows = merged[merged.duplicated(subset='UNITID', keep=False)]
duplicated_rows.head()

,UNITID,INSTNM,EIN,OPEID,IALIAS,ADDR,CITY,STABBR,ZIP,WEBADDR,...,YEARLYVAWA1K,IRS_MONTH,YEAR,PERCENTAGE_WOMEN_TRUSTEES,PERCENTAGE_WOMEN_KEY_EMPLOYEES,WHISTLEBLOWER_POLICY,CEO_REVIEWED_COMPENSATION,OTHER_REVIEWED_COMPENSATION,MALE_TO_FEMALE_PAY_RATIO,PRESIDENT_TO_AVERAGE_PAY_RATIO
28,101693,University of Mobile,630417508,102900,University of Mobile,5735 College Parkway,Mobile,AL,36613-2842,https://www.umobile.edu/,...,0,05A,2023.0,0.277778,0.333333,1.0,1.0,1.0,0.073269,7.500779
29,101693,University of Mobile,630417508,102900,University of Mobile,5735 College Parkway,Mobile,AL,36613-2842,https://www.umobile.edu/,...,0,05B,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,101912,Oakwood University,630366652,103300,OU,7000 Adventist Blvd NW,Huntsville,AL,35896,https://www2.oakwood.edu/,...,0.22675737,05A,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,101912,Oakwood University,630366652,103300,OU,7000 Adventist Blvd NW,Huntsville,AL,35896,https://www2.oakwood.edu/,...,0.22675737,05B,2023.0,0.187500,0.000000,1.0,1.0,1.0,0.000000,9.027479
38,102049,Samford University,630312914,103600,NaN,800 Lakeshore Drive,Birmingham,AL,35229-2240,www.samford.edu/,...,0,05A,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
duplicated_rows.shape

(609, 45)

In [4]:
duplicated_rows.to_csv("duplicated_rows.csv", index=False)

In [5]:
merged.shape

(2719, 45)

In [6]:
merged = merged.drop(duplicated_rows.index)

In [7]:
merged.shape

(2110, 45)

In [8]:
# Define the custom aggregation function
def resolve_duplicates(group):
    result = {}
    for col in group.columns:
        if col == 'UNITID':  # Keep UNITID as is
            result[col] = group[col].iloc[0]
        else:
            # Get all non-null values for the column
            non_null_values = group[col].dropna().values
            if len(non_null_values) == 0:
                result[col] = None  # No values
            elif len(non_null_values) == 1:
                result[col] = non_null_values[0]  # One unique value
            else:
                result[col] = non_null_values[-1]  # Latest value
    return pd.Series(result)

In [9]:
# Group by UNITID and apply the custom aggregation
result_df = duplicated_rows.groupby('UNITID').apply(resolve_duplicates).reset_index(drop=True)

<ipython-input-9-ec90a35408b1>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = duplicated_rows.groupby('UNITID').apply(resolve_duplicates).reset_index(drop=True)


In [10]:
result_df.head(10)

,UNITID,INSTNM,EIN,OPEID,IALIAS,ADDR,CITY,STABBR,ZIP,WEBADDR,...,YEARLYVAWA1K,IRS_MONTH,YEAR,PERCENTAGE_WOMEN_TRUSTEES,PERCENTAGE_WOMEN_KEY_EMPLOYEES,WHISTLEBLOWER_POLICY,CEO_REVIEWED_COMPENSATION,OTHER_REVIEWED_COMPENSATION,MALE_TO_FEMALE_PAY_RATIO,PRESIDENT_TO_AVERAGE_PAY_RATIO
0,101693,University of Mobile,630417508,102900,University of Mobile,5735 College Parkway,Mobile,AL,36613-2842,https://www.umobile.edu/,...,0,05B,2023.0,0.277778,0.333333,1.0,1.0,1.0,0.073269,7.500779
1,101912,Oakwood University,630366652,103300,OU,7000 Adventist Blvd NW,Huntsville,AL,35896,https://www2.oakwood.edu/,...,0.22675737,05B,2023.0,0.187500,0.000000,1.0,1.0,1.0,0.000000,9.027479
2,102049,Samford University,630312914,103600,None,800 Lakeshore Drive,Birmingham,AL,35229-2240,www.samford.edu/,...,0,05B,2023.0,NaN,NaN,1.0,1.0,1.0,NaN,14.238497
3,102094,University of South Alabama,630477348,105700,None,307 N University Blvd,Mobile,AL,36688-0002,www.southalabama.edu/,...,1.3890956,08A,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102234,Spring Hill College,630302179,104100,Springhill|Springhill College|SHC|the Hill,4000 Dauphin St,Mobile,AL,36608-1780,https://www.shc.edu/,...,1.274697259,05B,2023.0,0.407407,0.333333,1.0,1.0,1.0,0.435921,8.701081
5,102377,Tuskegee University,630288878,105000,None,"Kresge Center, 3rd Floor",Tuskegee,AL,36088-1920,www.tuskegee.edu/,...,0.518806744,10A,2023.0,0.333333,0.363636,1.0,1.0,0.0,1.255106,NaN
6,104586,Embry-Riddle Aeronautical University-Prescott,590936101,147902,Embry Riddle Aeronautical University-Prescott,3700 Willow Creek Road,Prescott,AZ,86301-3720,prescott.erau.edu/,...,0.210837023,05B,2023.0,0.111111,0.352941,1.0,1.0,1.0,1.763151,24.221998
7,105589,Prescott College,860294012,2065300,Prescot College | Prescott University | Presco...,220 Grove Ave,Prescott,AZ,86301,https://prescott.edu/,...,0.343997248,05B,2023.0,0.631579,0.666667,1.0,1.0,1.0,0.523765,8.744729
8,107044,Harding University,710236896,109700,None,915 E. Market Ave.,Searcy,AR,72149-5615,www.harding.edu/,...,0.208116545,05B,2023.0,0.142857,0.333333,1.0,1.0,0.0,2.320465,10.804253
9,107141,John Brown University,710239576,110000,None,2000 W University St,Siloam Springs,AR,72761,https://www.jbu.edu/,...,1.146460304,06A,2023.0,0.250000,0.125000,1.0,1.0,1.0,3.364066,26.385306


In [11]:
result_df.shape

(284, 45)

In [12]:
result_df.to_csv("deduplicated_rows.csv", index=False)

In [13]:
result = pd.concat([merged, result_df], ignore_index=True)
result.shape

(2394, 45)

In [16]:
result.to_csv("gf2025_final_deduplicated.csv", index=False)

In [17]:
dup = result[result.duplicated(subset='UNITID', keep=False)]
dup.head()

,UNITID,INSTNM,EIN,OPEID,IALIAS,ADDR,CITY,STABBR,ZIP,WEBADDR,...,YEARLYVAWA1K,IRS_MONTH,YEAR,PERCENTAGE_WOMEN_TRUSTEES,PERCENTAGE_WOMEN_KEY_EMPLOYEES,WHISTLEBLOWER_POLICY,CEO_REVIEWED_COMPENSATION,OTHER_REVIEWED_COMPENSATION,MALE_TO_FEMALE_PAY_RATIO,PRESIDENT_TO_AVERAGE_PAY_RATIO
